In [31]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

words = open('names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
char_to_int = {char: idx + 1 for idx, char in enumerate(chars)}
char_to_int['.'] = 0
int_to_char = {idx:char for char,idx in char_to_int.items()}

In [32]:
block_size = 3
X, Y = [], []

for w in words[:1]:

    context = [0]* block_size
    for ch in w + '.':
        ix = char_to_int[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [33]:
print(X.shape, X.dtype, Y.shape, Y.dtype)

print(X[0])
print(Y[0])

torch.Size([5, 3]) torch.int64 torch.Size([5]) torch.int64
tensor([0, 0, 0])
tensor(5)


In [34]:
C = torch.randn((27,2))
# rebuild X: same 5x3 slots, each id replaced by that row of C
emb = C[X]

C

tensor([[-2.2763e-01, -4.4058e-01],
        [ 3.9957e-01, -6.1890e-01],
        [-1.0034e-01, -1.1210e+00],
        [-1.1773e+00,  6.8211e-01],
        [-7.6386e-01,  2.3753e-01],
        [-1.2506e+00, -7.6894e-02],
        [ 1.2797e+00, -2.0259e-01],
        [-3.0372e-02, -1.8316e-01],
        [-2.4646e+00,  5.9852e-01],
        [ 4.7611e-01,  1.0142e+00],
        [ 5.5579e-02,  1.1042e-01],
        [-6.9616e-01, -1.0437e+00],
        [ 3.5850e-01,  3.0162e-01],
        [-6.9769e-01,  2.3718e-01],
        [-9.7843e-01,  5.9165e-01],
        [-1.0870e+00, -5.6322e-02],
        [ 2.8438e+00, -1.9632e-01],
        [-6.6499e-01,  3.1396e-01],
        [-6.5655e-02, -8.8879e-02],
        [-1.2618e+00, -1.8866e-02],
        [-2.0873e+00,  1.5116e-01],
        [-1.7266e+00, -2.7914e-03],
        [-1.6139e+00, -1.2385e-01],
        [-3.7546e-02, -4.3083e-02],
        [-1.1239e+00,  2.8933e-01],
        [ 1.2712e+00, -5.7482e-01],
        [-1.7248e+00,  1.6982e+00]])

In [35]:
emb.shape

torch.Size([5, 3, 2])

In [36]:
emb

tensor([[[-0.2276, -0.4406],
         [-0.2276, -0.4406],
         [-0.2276, -0.4406]],

        [[-0.2276, -0.4406],
         [-0.2276, -0.4406],
         [-1.2506, -0.0769]],

        [[-0.2276, -0.4406],
         [-1.2506, -0.0769],
         [-0.6977,  0.2372]],

        [[-1.2506, -0.0769],
         [-0.6977,  0.2372],
         [-0.6977,  0.2372]],

        [[-0.6977,  0.2372],
         [-0.6977,  0.2372],
         [ 0.3996, -0.6189]]])

`C[X]` **rebuilds `X`**. Same 5×3 slots. At each slot, take the integer in `X` and paste in that row of `C`. Layout stays `X`; contents become `C`'s rows. Each id is now a length-2 vector, so `(5, 3)` → `(5, 3, 2)`.

In [37]:
print('original X (ids):')
print(X)

# rebuild X: same slots, paste C's row for whatever id was there
x_rebuilt = torch.zeros((X.shape[0], X.shape[1], C.shape[1]))  # (5, 3, 2)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        id_in_x = X[i, j]             # integer currently in this slot of X
        x_rebuilt[i, j] = C[id_in_x]  # put that row of C in the same slot

print('X[1, 2] was', X[1, 2].item(), '-> pasted C[5] =', C[5])
print('same as C[X]?', torch.equal(x_rebuilt, C[X]))
x_rebuilt

original X (ids):
tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1]])
X[1, 2] was 5 -> pasted C[5] = tensor([-1.2506, -0.0769])
same as C[X]? True


tensor([[[-0.2276, -0.4406],
         [-0.2276, -0.4406],
         [-0.2276, -0.4406]],

        [[-0.2276, -0.4406],
         [-0.2276, -0.4406],
         [-1.2506, -0.0769]],

        [[-0.2276, -0.4406],
         [-1.2506, -0.0769],
         [-0.6977,  0.2372]],

        [[-1.2506, -0.0769],
         [-0.6977,  0.2372],
         [-0.6977,  0.2372]],

        [[-0.6977,  0.2372],
         [-0.6977,  0.2372],
         [ 0.3996, -0.6189]]])